# Strategy B: Handwriting ViT Encoder + Gated Attention Multiple Instance Learning (MIL)
### Universal Screening for Developmental Dysgraphia Without Page-Averaging Bloat

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com)

---

### Clinical Rationale & Architectural Design
1. **The Core Defect of Stage 21 (ImageNet DenseNet201)**:
   In Stage 21, features from an ImageNet CNN (trained on dogs and cars) were *averaged* across the child's entire page. This washes out dysgraphia: dysgraphic children typically write 15–20 readable words and suffer acute motor collapse on 3–5 difficult words or under fatigue.
2. **The Fix (Gated Attention MIL on Frozen Handwriting Foundation ViTs)**:
   - **Handwriting Foundation Encoders**: Instead of ImageNet textures, we use the frozen Vision Transformer encoders from `microsoft/trocr-small-handwritten` (English) and `paudelanil/trocr-devanagari-2` (Hindi), which naturally understand strokes, ligatures, loops, and ink curvature.
   - **Gated Attention Mechanism (Ilse et al., ICML 2018)**: An attention layer assigns an attention weight $a_k \in [0, 1]$ to each word on the page ($\sum a_k = 1$). It automatically learns to zoom in on the worst words on the page.
   - **Mathematical Immunity to Overfitting**: The vision encoders are 100% frozen. The attention layer and classification head contain only **~581 trainable parameters total**, making visual overfitting on our 215-student cohort mathematically impossible.
   - **Full Clinical Interpretability**: The attention weights $a_k$ rank every word on the page, highlighting the exact words that triggered the dysgraphia referral.


In [ ]:
# 1. Hardware Verification & Package Installation
!nvidia-smi

!pip install -q transformers torch torchvision scikit-learn matplotlib seaborn pandas numpy tqdm Pillow

import torch
print(f"PyTorch Version: {torch.__version__}")
print(f"CUDA Available: {torch.cuda.is_available()}")
if torch.cuda.is_available():
    print(f"Active GPU: {torch.cuda.get_device_name(0)}")
    device = "cuda"
else:
    print("WARNING: Running on CPU. Recommend switching to GPU in Runtime -> Change runtime type.")
    device = "cpu"


In [ ]:
# 2. Dataset Ingestion
import os
import shutil
import zipfile
from pathlib import Path
import pandas as pd

DATA_DIR = Path("word_crops_dataset")
ZIP_FILE = Path("word_crops_dataset.zip")

# Path A: Check if word_crops_dataset.zip is already uploaded or in Google Drive
if not DATA_DIR.exists():
    if not ZIP_FILE.exists():
        # Check Google Drive
        drive_path = Path("/content/drive/MyDrive/word_crops_dataset.zip")
        if drive_path.exists():
            print(f"Copying dataset from Google Drive: {drive_path}...")
            shutil.copy(drive_path, ZIP_FILE)
        else:
            print("Please upload 'word_crops_dataset.zip' (from your repository data/ folder)...")
            from google.colab import files
            uploaded = files.upload()

    print("Unzipping word_crops_dataset.zip...")
    with zipfile.ZipFile(ZIP_FILE, 'r') as zip_ref:
        zip_ref.extractall(DATA_DIR)
    print("Dataset extracted successfully!")

# Inspect Dataset Index
index_csv = DATA_DIR / "words_index.csv"
df_words = pd.read_csv(index_csv)
print(f"Total Word Crops: {len(df_words)}")
print(f"Unique Students: {df_words['student_id'].nunique()}")
print(f"Class Distribution: {df_words.groupby('label')['student_id'].nunique().to_dict()} (0=Control, 1=Dysgraphia)")
print(f"Script Breakdown: {df_words['script'].value_counts().to_dict()}")
df_words.head(3)


In [ ]:
# 3. Dual-ViT Feature Extraction (TrOCR English + Devanagari)
import csv
import torch
from PIL import Image
from transformers import TrOCRProcessor, VisionEncoderDecoderModel
from sklearn.decomposition import PCA
from tqdm.auto import tqdm
import numpy as np

CACHE_PATH = DATA_DIR / "cached_word_embeddings.pt"
PROJECT_DIM = 64

if CACHE_PATH.exists():
    print(f"Loading cached embeddings from {CACHE_PATH}...")
    cached = torch.load(CACHE_PATH, map_location="cpu", weights_only=False)
    proj_embeddings = cached["embeddings"]
    records = cached["records"]
    print("Embeddings loaded successfully!")
else:
    print("Starting Dual-ViT Feature Extraction across 16,469 words...")
    with open(index_csv, "r", encoding="utf-8") as f:
        records = list(csv.DictReader(f))

    latin_recs = [r for r in records if r["script"] == "latin"]
    hindi_recs = [r for r in records if r["script"] == "devanagari"]
    word_embeddings = {}

    # A. English ViT: microsoft/trocr-small-handwritten
    print(f"\n[1/2] Loading English TrOCR for {len(latin_recs)} words...")
    eng_proc = TrOCRProcessor.from_pretrained("microsoft/trocr-small-handwritten")
    eng_model = VisionEncoderDecoderModel.from_pretrained("microsoft/trocr-small-handwritten").to(device)
    eng_encoder = eng_model.encoder.eval()
    for p in eng_encoder.parameters():
        p.requires_grad = False

    batch_size = 32
    for i in tqdm(range(0, len(latin_recs), batch_size), desc="English ViT"):
        batch = latin_recs[i:i+batch_size]
        imgs = [Image.open(DATA_DIR / r["rel_path"]).convert("RGB") for r in batch if (DATA_DIR / r["rel_path"]).exists()]
        if not imgs:
            continue
        inputs = eng_proc(imgs, return_tensors="pt").pixel_values.to(device)
        with torch.no_grad():
            h = eng_encoder(inputs).last_hidden_state.mean(dim=1).cpu()
        for r, emb in zip(batch, h):
            word_embeddings[r["word_id"]] = emb

    del eng_model, eng_encoder, eng_proc
    if torch.cuda.is_available():
        torch.cuda.empty_cache()

    # B. Hindi ViT: paudelanil/trocr-devanagari-2
    print(f"\n[2/2] Loading Devanagari TrOCR for {len(hindi_recs)} words...")
    hin_proc = TrOCRProcessor.from_pretrained("paudelanil/trocr-devanagari-2")
    hin_model = VisionEncoderDecoderModel.from_pretrained("paudelanil/trocr-devanagari-2").to(device)
    hin_encoder = hin_model.encoder.eval()
    for p in hin_encoder.parameters():
        p.requires_grad = False

    for i in tqdm(range(0, len(hindi_recs), batch_size), desc="Hindi ViT"):
        batch = hindi_recs[i:i+batch_size]
        imgs = [Image.open(DATA_DIR / r["rel_path"]).convert("RGB") for r in batch if (DATA_DIR / r["rel_path"]).exists()]
        if not imgs:
            continue
        inputs = hin_proc(imgs, return_tensors="pt").pixel_values.to(device)
        with torch.no_grad():
            h = hin_encoder(inputs).last_hidden_state.mean(dim=1).cpu()
        for r, emb in zip(batch, h):
            word_embeddings[r["word_id"]] = emb

    del hin_model, hin_encoder, hin_proc
    if torch.cuda.is_available():
        torch.cuda.empty_cache()

    # C. Project to Shared Low-Rank Dimension (d=64)
    print(f"\nProjecting ViT embeddings into shared latent space (d={PROJECT_DIM})...")
    dim_groups = {}
    for k, v in word_embeddings.items():
        dim_groups.setdefault(v.shape[0], []).append(k)

    proj_embeddings = {}
    for d_in, keys in dim_groups.items():
        mat = torch.stack([word_embeddings[k] for k in keys]).numpy()
        pca = PCA(n_components=min(PROJECT_DIM, d_in, mat.shape[0]))
        p_mat = pca.fit_transform(mat)
        if p_mat.shape[1] < PROJECT_DIM:
            p_mat = np.hstack([p_mat, np.zeros((p_mat.shape[0], PROJECT_DIM - p_mat.shape[1]))])
        for idx, k in enumerate(keys):
            proj_embeddings[k] = torch.tensor(p_mat[idx], dtype=torch.float32)

    torch.save({"embeddings": proj_embeddings, "records": records}, CACHE_PATH)
    print(f"Cached embeddings saved to {CACHE_PATH}!")


In [ ]:
# 4. Gated Attention MIL Architecture (~581 Trainable Parameters)
import torch.nn as nn

class GatedAttention(nn.Module):
    """Ilse et al. (ICML 2018) Gated Attention Pooling Layer."""
    def __init__(self, in_features=64, attention_dim=4):
        super().__init__()
        self.attention_v = nn.Linear(in_features, attention_dim, bias=False)
        self.attention_u = nn.Linear(in_features, attention_dim, bias=False)
        self.attention_w = nn.Linear(attention_dim, 1, bias=False)
        nn.init.xavier_uniform_(self.attention_v.weight)
        nn.init.xavier_uniform_(self.attention_u.weight)
        nn.init.xavier_uniform_(self.attention_w.weight)

    def forward(self, h, mask=None):
        # h: (batch_size, num_instances, in_features)
        v = torch.tanh(self.attention_v(h))
        u = torch.sigmoid(self.attention_u(h))
        scores = self.attention_w(v * u) # (batch_size, num_instances, 1)

        if mask is not None:
            scores = scores.squeeze(-1)
            scores = scores.masked_fill(~mask, -1e9)
            weights = torch.softmax(scores, dim=-1).unsqueeze(-1)
        else:
            weights = torch.softmax(scores, dim=1)

        z = torch.sum(weights * h, dim=1) # (batch_size, in_features)
        return z, weights.squeeze(-1)

class GatedAttentionMIL(nn.Module):
    def __init__(self, encoder_dim=64, projected_dim=64, attention_dim=4, dropout=0.15):
        super().__init__()
        self.projection = nn.Identity() if encoder_dim == projected_dim else nn.Linear(encoder_dim, projected_dim)
        self.attention = GatedAttention(in_features=projected_dim, attention_dim=attention_dim)
        self.classifier = nn.Sequential(
            nn.Dropout(dropout),
            nn.Linear(projected_dim, 1)
        )
        if isinstance(self.classifier[-1], nn.Linear):
            nn.init.xavier_uniform_(self.classifier[-1].weight)
            nn.init.zeros_(self.classifier[-1].bias)

    def forward(self, h, mask=None):
        proj = self.projection(h)
        z, attn_weights = self.attention(proj, mask=mask)
        logits = self.classifier(z).squeeze(-1)
        return logits, attn_weights

    def count_params(self):
        return sum(p.numel() for p in self.parameters() if p.requires_grad)

sample_model = GatedAttentionMIL(encoder_dim=64, projected_dim=64, attention_dim=4)
print(f"Total Trainable Parameters in Gated-Attention MIL: {sample_model.count_params()} (Strictly constrained << 1,000!)")


In [ ]:
# 5. Locked 5-Repeat x 5-Fold Stratified Group Cross-Validation
from torch.utils.data import Dataset, DataLoader
from sklearn.metrics import roc_auc_score, average_precision_score, brier_score_loss, roc_curve

# Group word embeddings by student
student_bags = {}
student_meta = {}
student_words = {}

for r in records:
    sid = r["student_id"]
    wid = r["word_id"]
    if wid in proj_embeddings:
        student_words.setdefault(sid, []).append((r, proj_embeddings[wid]))
        if sid not in student_meta:
            student_meta[sid] = {
                "school": r["school"],
                "grade": int(r["grade"]),
                "label": int(r["label"]),
                "repeat_0_fold": int(r["repeat_0_fold"]),
                "repeat_1_fold": int(r["repeat_1_fold"]),
                "repeat_2_fold": int(r["repeat_2_fold"]),
                "repeat_3_fold": int(r["repeat_3_fold"]),
                "repeat_4_fold": int(r["repeat_4_fold"]),
            }

for sid, pairs in student_words.items():
    student_bags[sid] = torch.stack([p[1] for p in pairs])

all_sids = sorted(list(student_bags.keys()))
n_pos = sum(1 for sid in all_sids if student_meta[sid]["label"] == 1)
n_neg = len(all_sids) - n_pos
pos_weight = float(n_neg / max(1, n_pos))
print(f"Cohort Assembly: {len(all_sids)} students (40 Dysgraphic, 175 Control | Class Weight = {pos_weight:.2f})")

def collate_fn(batch):
    max_k = max(item["features"].shape[0] for item in batch)
    d = batch[0]["features"].shape[1]
    padded = torch.zeros((len(batch), max_k, d))
    mask = torch.zeros((len(batch), max_k), dtype=torch.bool)
    labels = torch.zeros((len(batch),))
    sids = [item["student_id"] for item in batch]
    for i, it in enumerate(batch):
        k = it["features"].shape[0]
        padded[i, :k, :] = it["features"]
        mask[i, :k] = True
        labels[i] = it["label"]
    return {"features": padded, "mask": mask, "labels": labels, "student_ids": sids}

class SimpleBagDS(Dataset):
    def __init__(self, sids):
        self.sids = sids
    def __len__(self):
        return len(self.sids)
    def __getitem__(self, idx):
        sid = self.sids[idx]
        return {"features": student_bags[sid], "label": float(student_meta[sid]["label"]), "student_id": sid}

# Run 5-Repeat x 5-Fold Evaluation
repeat_aurocs = []
repeat_auprcs = []
oof_preds = {sid: [] for sid in all_sids}
sample_attentions = {}

EPOCHS = 35
LR = 1.2e-3
WEIGHT_DECAY = 1e-3

for r in range(5):
    fold_probs = {}
    for f in range(5):
        train_sids = [sid for sid in all_sids if student_meta[sid][f"repeat_{r}_fold"] != f]
        val_sids = [sid for sid in all_sids if student_meta[sid][f"repeat_{r}_fold"] == f]

        train_loader = DataLoader(SimpleBagDS(train_sids), batch_size=8, shuffle=True, collate_fn=collate_fn)
        val_loader = DataLoader(SimpleBagDS(val_sids), batch_size=8, shuffle=False, collate_fn=collate_fn)

        model = GatedAttentionMIL(encoder_dim=64, projected_dim=64, attention_dim=4, dropout=0.15).to(device)
        criterion = nn.BCEWithLogitsLoss(pos_weight=torch.tensor([pos_weight], device=device))
        optimizer = torch.optim.AdamW(model.parameters(), lr=LR, weight_decay=WEIGHT_DECAY)
        scheduler = torch.optim.lr_scheduler.CosineAnnealingLR(optimizer, T_max=EPOCHS, eta_min=1e-5)

        best_loss = float("inf")
        best_fold_preds = {}

        for epoch in range(EPOCHS):
            model.train()
            for batch in train_loader:
                feats = batch["features"].to(device)
                mask = batch["mask"].to(device)
                lbls = batch["labels"].to(device)
                optimizer.zero_grad()
                logits, _ = model(feats, mask=mask)
                loss = criterion(logits, lbls)
                loss.backward()
                optimizer.step()
            scheduler.step()

            model.eval()
            val_loss = 0.0
            curr_preds = {}
            with torch.no_grad():
                for batch in val_loader:
                    feats = batch["features"].to(device)
                    mask = batch["mask"].to(device)
                    lbls = batch["labels"].to(device)
                    logits, attns = model(feats, mask=mask)
                    loss = criterion(logits, lbls)
                    val_loss += loss.item() * len(lbls)
                    probs = torch.sigmoid(logits).cpu().numpy()
                    attns_np = attns.cpu().numpy()
                    for sid_b, p_b, a_b, m_b in zip(batch["student_ids"], probs, attns_np, mask.cpu().numpy()):
                        curr_preds[sid_b] = float(p_b)
                        if r == 0:
                            sample_attentions[sid_b] = a_b[m_b]

            val_loss /= max(1, len(val_sids))
            if val_loss < best_loss:
                best_loss = val_loss
                best_fold_preds = curr_preds

        fold_probs.update(best_fold_preds)

    y_t = [student_meta[sid]["label"] for sid in all_sids]
    y_p = [fold_probs[sid] for sid in all_sids]
    rep_auc = roc_auc_score(y_t, y_p)
    rep_pr = average_precision_score(y_t, y_p)
    repeat_aurocs.append(rep_auc)
    repeat_auprcs.append(rep_pr)
    print(f"Repeat {r+1}/5 Benchmark: AUROC = {rep_auc:.4f} | AUPRC = {rep_pr:.4f}")
    for sid in all_sids:
        oof_preds[sid].append(fold_probs[sid])

mean_oof = {sid: float(np.mean(oof_preds[sid])) for sid in all_sids}
y_true = np.array([student_meta[sid]["label"] for sid in all_sids])
y_score = np.array([mean_oof[sid] for sid in all_sids])

final_auroc = roc_auc_score(y_true, y_score)
final_auprc = average_precision_score(y_true, y_score)
final_brier = brier_score_loss(y_true, y_score)

fpr, tpr, _ = roc_curve(y_true, y_score)
sens_80 = tpr[np.where(fpr <= 0.20)[0][-1]] if np.any(fpr <= 0.20) else 0.0
sens_90 = tpr[np.where(fpr <= 0.10)[0][-1]] if np.any(fpr <= 0.10) else 0.0

print("\n" + "="*65)
print("FINAL STRATEGY B (Gated-Attention MIL) CLINICAL SCOREBOARD:")
print(f"  • AUROC: {final_auroc:.4f} (5-Repeat Mean: {np.mean(repeat_aurocs):.4f} ± {np.std(repeat_aurocs):.4f})")
print(f"  • AUPRC: {final_auprc:.4f} (5-Repeat Mean: {np.mean(repeat_auprcs):.4f} ± {np.std(repeat_auprcs):.4f})")
print(f"  • Brier Loss: {final_brier:.4f}")
print(f"  • Sensitivity @ 80% Specificity: {sens_80:.1%}")
print(f"  • Sensitivity @ 90% Specificity: {sens_90:.1%}")
print("="*65)


In [ ]:
# 6. Benchmark Visualization (ROC & PR Curves)
import matplotlib.pyplot as plt
from sklearn.metrics import precision_recall_curve

fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(14, 5), dpi=120)

# ROC Curve
fpr, tpr, _ = roc_curve(y_true, y_score)
ax1.plot(fpr, tpr, color='#1f77b4', lw=2.5, label=f'Gated-Attention MIL (AUROC = {final_auroc:.4f})')
ax1.plot([0, 1], [0, 1], color='gray', lw=1, linestyle='--')
ax1.axvline(0.10, color='crimson', linestyle=':', label='90% Specificity Operating Point')
ax1.set_xlabel('False Positive Rate (1 - Specificity)', fontsize=11)
ax1.set_ylabel('True Positive Rate (Sensitivity)', fontsize=11)
ax1.set_title('Receiver Operating Characteristic (ROC)', fontsize=12, fontweight='bold')
ax1.legend(loc='lower right', frameon=True)
ax1.grid(True, alpha=0.3)

# Precision-Recall Curve
prec, rec, _ = precision_recall_curve(y_true, y_score)
no_skill = n_pos / len(all_sids)
ax2.plot(rec, prec, color='#2ca02c', lw=2.5, label=f'Gated-Attention MIL (AUPRC = {final_auprc:.4f})')
ax2.axhline(no_skill, color='gray', lw=1, linestyle='--', label=f'Random Chance ({no_skill:.2f})')
ax2.set_xlabel('Recall (Sensitivity)', fontsize=11)
ax2.set_ylabel('Precision (Positive Predictive Value)', fontsize=11)
ax2.set_title('Precision-Recall (PR) Curve', fontsize=12, fontweight='bold')
ax2.legend(loc='upper right', frameon=True)
ax2.grid(True, alpha=0.3)

plt.tight_layout()
plt.show()


In [ ]:
# 7. Clinical Attention Interpretability Gallery
# Visualizes the top-3 words that the Gated-Attention layer attended to
import matplotlib.pyplot as plt

pos_students = [sid for sid in all_sids if student_meta[sid]['label'] == 1 and sid in sample_attentions]
neg_students = [sid for sid in all_sids if student_meta[sid]['label'] == 0 and sid in sample_attentions]

def inspect_student_attention(sid, title_prefix=""):
    attns = sample_attentions[sid]
    word_pairs = student_words[sid]
    # Sort words by attention score
    ranked_indices = np.argsort(attns)[::-1]
    
    print(f"\n{'='*70}\n{title_prefix} Student: {sid} (True Label={student_meta[sid]['label']}, Pred Risk={mean_oof[sid]:.1%})\n{'='*70}")
    top_indices = ranked_indices[:3]
    fig, axes = plt.subplots(1, min(3, len(top_indices)), figsize=(12, 3), dpi=120)
    if len(top_indices) == 1:
        axes = [axes]
    for ax, idx in zip(axes, top_indices):
        rec, _ = word_pairs[idx]
        img_p = DATA_DIR / rec['rel_path']
        if img_p.exists():
            img = Image.open(img_p)
            ax.imshow(img)
            ax.set_title(f"Attention Weight: {attns[idx]:.3f}\nTask: {rec['task_id']}", fontsize=10)
        ax.axis('off')
    plt.suptitle(f"Top-3 Attended Words for {sid}", fontsize=12, fontweight='bold', y=1.05)
    plt.show()

# Show 2 Dysgraphic Students and 2 Control Students
if pos_students:
    inspect_student_attention(pos_students[0], "[Dysgraphic Positive]")
if len(pos_students) > 1:
    inspect_student_attention(pos_students[1], "[Dysgraphic Positive]")
if neg_students:
    inspect_student_attention(neg_students[0], "[Neurotypical Control]")


In [ ]:
# 8. Export Out-of-Fold Predictions for Late Stacking
export_df = pd.DataFrame([
    {
        "student_id": sid,
        "school": student_meta[sid]["school"],
        "grade": student_meta[sid]["grade"],
        "label": student_meta[sid]["label"],
        "pred_prob_mil": mean_oof[sid],
        "repeat_0_prob": oof_preds[sid][0] if len(oof_preds[sid]) > 0 else mean_oof[sid],
        "repeat_1_prob": oof_preds[sid][1] if len(oof_preds[sid]) > 1 else mean_oof[sid],
        "repeat_2_prob": oof_preds[sid][2] if len(oof_preds[sid]) > 2 else mean_oof[sid],
        "repeat_3_prob": oof_preds[sid][3] if len(oof_preds[sid]) > 3 else mean_oof[sid],
        "repeat_4_prob": oof_preds[sid][4] if len(oof_preds[sid]) > 4 else mean_oof[sid],
    }
    for sid in all_sids
])

export_path = "p_mil_vit_predictions.csv"
export_df.to_csv(export_path, index=False)
print(f"Exported out-of-fold predictions to {export_path} ({len(export_df)} students)!")

try:
    from google.colab import files
    files.download(export_path)
    print("Download triggered automatically.")
except Exception:
    print(f"File saved locally at: {export_path}")
